In [0]:
%run "./common functions"

In [0]:
dbutils.notebook.run("./LinkedIn/getting linkedin jobs", 0)

In [0]:
df = spark.read.json("/Volumes/raw_catalogue/jobs/vol/jobs_from_github/")
df.createOrReplaceTempView("source_vw")

In [0]:
%sql
create or replace table staging_catalogue.jobs.all_jobs as
select * from (
select title, company, url, description, location from source_vw
union 
select title, company, url, description, location from raw_catalogue.jobs.linkedJobs) a
left anti join main_catalogue.jobs.all_jobs b
on a.url = b.url

In [0]:
scored_df = (spark.sql(f"""select ai_query("databricks-llama-4-maverick", concat('{p1}', 'Company Name:', company, description, '{p2}', '{format}', '{rules}')) as cl,* from staging_catalogue.jobs.all_jobs""")
             .withColumn("score", get_json_object("cl", "$.score").cast("int")) 
            .withColumn("company_type", get_json_object("cl", "$.company_type"))
            .drop("cl", "description", "source"))

In [0]:
filtered_df = scored_df.filter(scored_df.score >= 70).orderBy(col("score").desc())
filtered_df.write.mode("overwrite").saveAsTable("staging_catalogue.jobs.naukari_linkedin")

In [0]:
form_message_and_send("staging_catalogue.jobs.naukari_linkedin")

In [0]:
filtered_df.write.mode("append").saveAsTable("main_catalogue.jobs.all_jobs")